In [ ]:
import pandas as pd
import sqlite3

keywords = ["cathode", "LFP", "NMC", "lithium iron phosphate", "nickel manganese cobalt", 
            "lithium battery", "anode", "electrolyte"]

conn = sqlite3.connect("patents.db")

print("正在处理 g_patent.tsv ...")
matched_patents = []

for chunk in pd.read_csv("g_patent.tsv", sep="\t", chunksize=50000, 
                          on_bad_lines='skip', low_memory=False):
    if 'patent_date' in chunk.columns:
        chunk = chunk[(chunk['patent_date'] >= '2018-01-01') & 
                      (chunk['patent_date'] <= '2024-07-31')]
    if 'patent_title' in chunk.columns:
        mask = chunk['patent_title'].str.lower().str.contains(
            '|'.join(keywords), na=False)
        matched_patents.append(chunk[mask])

df_patents = pd.concat(matched_patents, ignore_index=True)
print(f"匹配专利数: {len(df_patents)}")
print(df_patents.columns.tolist())
df_patents.to_sql("patents", conn, if_exists="replace", index=False)
print("已存入 patents 表")

正在处理 g_patent.tsv ...
匹配专利数: 6628
['patent_id', 'patent_type', 'patent_date', 'patent_title', 'wipo_kind', 'num_claims', 'withdrawn', 'filename']
已存入 patents 表


In [3]:
print("正在处理 g_patent_abstract.tsv ...")

# 只取已匹配的 patent_id
matched_ids = set(df_patents['patent_id'].astype(str).tolist())

matched_abstracts = []

for chunk in pd.read_csv("g_patent_abstract.tsv", sep="\t", chunksize=50000,
                          on_bad_lines='skip', low_memory=False):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    matched = chunk[chunk['patent_id'].isin(matched_ids)]
    if len(matched) > 0:
        matched_abstracts.append(matched)

df_abstracts = pd.concat(matched_abstracts, ignore_index=True)
print(f"匹配摘要数: {len(df_abstracts)}")
print(df_abstracts.columns.tolist())
df_abstracts.to_sql("abstracts", conn, if_exists="replace", index=False)
print("已存入 abstracts 表")

正在处理 g_patent_abstract.tsv ...
匹配摘要数: 6628
['patent_id', 'patent_abstract']
已存入 abstracts 表


In [4]:
print("正在处理 g_assignee_disambiguated.tsv ...")

matched_ids = set(df_patents['patent_id'].astype(str).tolist())
matched_assignees = []

for chunk in pd.read_csv("g_assignee_disambiguated.tsv", sep="\t", chunksize=50000,
                         on_bad_lines='skip', low_memory=False):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    matched = chunk[chunk['patent_id'].isin(matched_ids)]
    if len(matched) > 0:
        matched_assignees.append(matched)

df_assignee = pd.concat(matched_assignees, ignore_index=True)
print(f"匹配机构数: {len(df_assignee)}")
print(df_assignee.columns.tolist())
df_assignee.to_sql("assignee", conn, if_exists="replace", index=False)
print("已存入 assignee 表")

正在处理 g_assignee_disambiguated.tsv ...
匹配机构数: 7390
['patent_id', 'assignee_sequence', 'assignee_id', 'disambig_assignee_individual_name_first', 'disambig_assignee_individual_name_last', 'disambig_assignee_organization', 'assignee_type', 'location_id']
已存入 assignee 表


In [5]:
print("正在处理 g_cpc_current.tsv ...")

matched_ids = set(df_patents['patent_id'].astype(str).tolist())
matched_cpc = []

for chunk in pd.read_csv("g_cpc_current.tsv", sep="\t", chunksize=50000,
                         on_bad_lines='skip', low_memory=False):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    matched = chunk[chunk['patent_id'].isin(matched_ids)]
    if len(matched) > 0:
        matched_cpc.append(matched)

df_cpc = pd.concat(matched_cpc, ignore_index=True)
print(f"匹配CPC数: {len(df_cpc)}")
print(df_cpc.columns.tolist())
df_cpc.to_sql("cpc", conn, if_exists="replace", index=False)
print("已存入 cpc 表")

正在处理 g_cpc_current.tsv ...
匹配CPC数: 80946
['patent_id', 'cpc_sequence', 'cpc_section', 'cpc_class', 'cpc_subclass', 'cpc_group', 'cpc_type']
已存入 cpc 表


In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("patents.db")

tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

print(tables)

        name
0  abstracts
1   assignee
2        cpc
3    patents
